# Multi-Crop Leaf Disease Detection - Google Colab Training + Quantization (Drive Dataset)

**This notebook trains MobileNetV2 and EfficientNet-Lite0 models and exports TFLite (dynamic + full INT8) using Google Drive for dataset and model storage.**

**Prerequisites:**
- Runtime -> Change runtime type -> GPU
- Dataset in Google Drive at `/MyDrive/leaf_data/processed` with train/val/test splits
- Google Drive mounted (for dataset + saving models and TFLite exports)

**Key differences:**
- Dataset is read directly from Google Drive (no upload to `/content`)
- Models and TFLite exports saved to Google Drive
- Optional evaluation and quantization steps are included

## Step 1: Mount Google Drive (dataset + model storage)

In [ ]:
from google.colab import drive  # pyright: ignore[reportMissingImports]
drive.mount('/content/drive')
print("✓ Google Drive mounted (dataset + models)")
print("Note: Dataset is read directly from Drive")

## Step 2: Use Dataset from Google Drive

In [ ]:
import os

drive_dataset = "/content/drive/MyDrive/leaf_data/processed"

if os.path.exists(drive_dataset):
    print("Found Drive dataset at:", drive_dataset)

    # Quick listing to confirm expected splits
    for split in ["train", "val", "test"]:
        split_path = os.path.join(drive_dataset, split)
        if os.path.exists(split_path):
            n_classes = len([d for d in os.listdir(split_path) if os.path.isdir(os.path.join(split_path, d))])
            print(f"  ✓ {split}: {n_classes} classes")
        else:
            print(f"  ✗ {split}: NOT FOUND")
else:
    print("Drive dataset not found at", drive_dataset)

## Step 3: Clone Project Repository

In [ ]:
import os

%cd /content

# Clone repo (replace with your GitHub repo URL)
repo_dir = "/content/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System"
if not os.path.exists(repo_dir):
    !git clone https://github.com/hit1363/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System.git

%cd {repo_dir}
print("Repository ready!")

## Step 4: Install Dependencies

In [ ]:
%pip install -q -r requirements.txt
print("Dependencies installed successfully!")

## Step 5: Configure Training for MobileNetV2 (2-Phase Schedule)

In [ ]:
import yaml
import os

# Paths - Dataset read from Drive, models save to Drive
repo_dir = "/content/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System"
dataset_base = "/content/drive/MyDrive/leaf_data/processed"
output_base = "/content/drive/MyDrive/leaf_models"
os.makedirs(output_base, exist_ok=True)

# Load config
cfg_path = os.path.join(repo_dir, "training", "config_mobilenetv2.yaml")
with open(cfg_path, "r") as f:
    cfg = yaml.safe_load(f)

# Set dataset paths (from Drive)
cfg["dataset"]["data_dir"] = dataset_base
cfg["dataset"]["train_dir"] = f"{dataset_base}/train"
cfg["dataset"]["val_dir"] = f"{dataset_base}/val"
cfg["dataset"]["test_dir"] = f"{dataset_base}/test"

# Auto-count classes from training split
num_classes = len([
    d for d in os.listdir(cfg["dataset"]["train_dir"])
    if os.path.isdir(os.path.join(cfg["dataset"]["train_dir"], d))
])
cfg["model"]["num_classes"] = num_classes

# Training schedule: 5 epochs frozen + 20 epochs fine-tune
cfg["training"]["epochs"] = 25
cfg["training"]["freeze_base"] = True
cfg["training"]["unfreeze_epoch"] = 5
cfg["training"]["freeze_until_layer"] = 50
cfg["training"]["fine_tune_learning_rate"] = 0.0001

# Optimizer + LR schedule (ReduceLROnPlateau supported in training script)
cfg["optimizer"]["learning_rate"] = 0.001
cfg["lr_schedule"]["type"] = "reduce_on_plateau"
cfg["lr_schedule"]["monitor"] = "val_loss"
cfg["lr_schedule"]["factor"] = 0.5
cfg["lr_schedule"]["patience"] = 5
cfg["lr_schedule"]["min_lr"] = 1e-7

# Set output paths - Models ONLY to Drive (skip logs to save space)
cfg["export"]["save_dir"] = f"{output_base}/mobilenetv2"

# Disable TensorBoard logging to save space
cfg["callbacks"]["tensorboard"]["enabled"] = False
cfg["callbacks"]["csv_logger"]["filename"] = f"{output_base}/training_log_mobilenetv2.csv"

# Save updated config
with open(cfg_path, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("✓ Configuration updated (Drive dataset)")
print("✓ Training schedule: 5 epochs frozen + 20 epochs fine-tune")
print("✓ Unfreeze from layer: 50")
print(f"✓ Classes: {cfg['model']['num_classes']}")
print(f"✓ Train dataset: {cfg['dataset']['train_dir']}")
print(f"✓ Models save to: {cfg['export']['save_dir']}")
print("✓ TensorBoard logs: DISABLED (saves space)")

## Step 6: Train MobileNetV2

In [ ]:
%cd /content/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System/training

!python train.py --config config_mobilenetv2.yaml

## Step 7: Configure & Train EfficientNet-Lite0 (2-Phase Schedule)

In [ ]:
import yaml
import os

# Load EfficientNet config
repo_dir = "/content/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System"
cfg_path = os.path.join(repo_dir, "training", "config_efficientnet_lite0.yaml")
with open(cfg_path, "r") as f:
    cfg = yaml.safe_load(f)

# Paths
dataset_base = "/content/drive/MyDrive/leaf_data/processed"
output_base = "/content/drive/MyDrive/leaf_models"
os.makedirs(output_base, exist_ok=True)

# Set dataset paths (from Drive)
cfg["dataset"]["data_dir"] = dataset_base
cfg["dataset"]["train_dir"] = f"{dataset_base}/train"
cfg["dataset"]["val_dir"] = f"{dataset_base}/val"
cfg["dataset"]["test_dir"] = f"{dataset_base}/test"

# Auto-count classes
num_classes = len([
    d for d in os.listdir(cfg["dataset"]["train_dir"])
    if os.path.isdir(os.path.join(cfg["dataset"]["train_dir"], d))
])
cfg["model"]["num_classes"] = num_classes

# Training schedule: 5 epochs frozen + 20 epochs fine-tune
cfg["training"]["epochs"] = 25
cfg["training"]["freeze_base"] = True
cfg["training"]["unfreeze_epoch"] = 5
cfg["training"]["freeze_until_layer"] = 50
cfg["training"]["fine_tune_learning_rate"] = 0.0001

# Optimizer + LR schedule (ReduceLROnPlateau supported in training script)
cfg["optimizer"]["learning_rate"] = 0.001
cfg["lr_schedule"]["type"] = "reduce_on_plateau"
cfg["lr_schedule"]["monitor"] = "val_loss"
cfg["lr_schedule"]["factor"] = 0.5
cfg["lr_schedule"]["patience"] = 5
cfg["lr_schedule"]["min_lr"] = 1e-7

# Set output paths
cfg["export"]["save_dir"] = f"{output_base}/efficientnet_lite0"

# Disable TensorBoard logging to save space
cfg["callbacks"]["tensorboard"]["enabled"] = False
cfg["callbacks"]["csv_logger"]["filename"] = f"{output_base}/training_log_efficientnet_lite0.csv"

# Save updated config
with open(cfg_path, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("✓ EfficientNet-Lite0 config updated (Drive dataset)")
print("✓ Training schedule: 5 epochs frozen + 20 epochs fine-tune")
print("✓ Unfreeze from layer: 50")
print(f"✓ Classes: {cfg['model']['num_classes']}")
print(f"✓ Models save to: {cfg['export']['save_dir']}")

In [ ]:
!python train.py --config config_efficientnet_lite0.yaml

## Step 8: List Trained Models and Exports

In [ ]:
import os

models_dir = "/content/drive/MyDrive/leaf_models"

print("Trained models saved to Google Drive:\n")

for arch in ["mobilenetv2", "efficientnet_lite0"]:
    arch_dir = os.path.join(models_dir, arch)
    if os.path.exists(arch_dir):
        print(f"{arch}:")
        for f in sorted(os.listdir(arch_dir)):
            fpath = os.path.join(arch_dir, f)
            if os.path.isfile(fpath):
                fsize = os.path.getsize(fpath) / (1024**2)
                print(f"  - {f} ({fsize:.1f} MB)")
            else:
                print(f"  - {f}/ (dir)")
    else:
        print(f"{arch}: No models found yet")

## Step 9: Evaluate a Trained Model (MobileNetV2 Example)

In [ ]:
import os
import glob
import subprocess

# Choose architecture to evaluate
arch = "mobilenetv2"  # or "efficientnet_lite0"
models_dir = f"/content/drive/MyDrive/leaf_models/{arch}"

def find_latest_model(arch_dir):
    saved_models = sorted(
        glob.glob(os.path.join(arch_dir, "saved_model_*")),
        key=os.path.getmtime,
    )
    if saved_models:
        return saved_models[-1]
    h5_models = sorted(
        glob.glob(os.path.join(arch_dir, "*.h5")),
        key=os.path.getmtime,
    )
    if h5_models:
        return h5_models[-1]
    return None

if os.path.exists(models_dir):
    model_path = find_latest_model(models_dir)
    if model_path:
        print(f"Evaluating: {os.path.basename(model_path)}\n")
        training_dir = "/content/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System/training"
        os.chdir(training_dir)
        config_name = (
            "config_mobilenetv2.yaml"
            if arch == "mobilenetv2"
            else "config_efficientnet_lite0.yaml"
        )
        subprocess.run(["python", "evaluate.py", "--model", model_path, "--config", config_name])
    else:
        print("No model found for evaluation")
else:
    print(f"Directory not found: {models_dir}")

## Step 13: View Training Logs from Drive

In [ ]:
import pandas as pd
import os

results_dir = "/content/drive/MyDrive/leaf_models"

if os.path.exists(results_dir):
    csv_files = [f for f in os.listdir(results_dir) if f.endswith('.csv')]
    if csv_files:
        for csv_file in csv_files:
            csv_path = os.path.join(results_dir, csv_file)
            print(f"\n=== {csv_file} ===")
            df = pd.read_csv(csv_path)
            print(df.tail(10))  # Show last 10 rows
    else:
        print("No CSV files found yet")
else:
    print("Results directory not found yet")

## Step 14: Download Trained Models to Computer

In [ ]:
import os
import shutil
from google.colab import files  # pyright: ignore[reportMissingImports]

models_dir = "/content/drive/MyDrive/leaf_models"

print("=" * 60)
print("DOWNLOAD TRAINED MODELS")
print("=" * 60)

# Create a zip of all models
if os.path.exists(models_dir):
    print("\nPreparing models for download...")
    shutil.make_archive('leaf_models', 'zip', models_dir)
    
    # Show file sizes
    for arch in ['mobilenetv2', 'efficientnet_lite0']:
        arch_dir = os.path.join(models_dir, arch)
        if os.path.exists(arch_dir):
            for f in os.listdir(arch_dir):
                fpath = os.path.join(arch_dir, f)
                if os.path.isfile(fpath):
                    size = os.path.getsize(fpath) / (1024**2)
                    print(f"  {f}: {size:.1f} MB")
    
    print("\n✓ Downloading leaf_models.zip...")
    files.download('leaf_models.zip')
    print("✓ Download complete! Save this to your models/ folder")
else:
    print("No models found to download")

## Storage Summary

**What happened:**
- ✓ Dataset read directly from Google Drive (`/My Drive/leaf_data/processed`)
- ✓ Models trained and saved to Google Drive (`/My Drive/leaf_models/`)
- ✓ Quantized TFLite models saved to Drive (`/My Drive/leaf_models/quantized/`)
- ✓ Downloaded final models to your computer

**Google Drive usage:**
- Dataset stays on Drive (no upload to Colab storage)
- Added model checkpoints, logs, and quantized TFLite files

**Next steps:**
1. Extract `leaf_models.zip` to your `models/` folder
2. Use the INT8 TFLite model in your Flutter app for best on-device speed
3. Or, deploy models to your server for remote inference

**For future training:**
- Keep this notebook template
- No dataset upload needed; just reuse the Drive folder
- Old models stay on Drive as backups